### 1. Install Required Libraries & Mount Google Drive

In [1]:
# Install necessary python libraries
!pip install python-docx beautifulsoup4 requests pillow lxml --quiet

# Mount Google Drive
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
    output_folder = '/content/drive/My Drive/URL_Downloads'
    os.makedirs(output_folder, exist_ok=True)
    print(f"Successfully set up output directory: {output_folder}")
except Exception as e:
    print(f"Error mounting Drive: {e}. Documents will be saved locally if Drive is unavailable.")
    output_folder = '/content/URL_Downloads'
    os.makedirs(output_folder, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully set up output directory: /content/drive/My Drive/URL_Downloads


### 2. URL to DOCX Downloader Script
This script parses the HTML structure element by element to preserve order. It handles headings, paragraphs, tables, images, and right-to-left text alignment.

In [5]:
import os
import re
import io
import requests
from bs4 import BeautifulSoup, NavigableString
from urllib.parse import urljoin, urlparse
from PIL import Image
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

def set_rtl(paragraph):
    """Helper to apply Right-to-Left (RTL) alignment and properties to a paragraph."""
    paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    pPr = paragraph._p.get_or_add_pPr()
    bidi = OxmlElement('w:bidi')
    bidi.set(qn('w:val'), '1')
    pPr.append(bidi)

def is_rtl_text(text):
    """Simple heuristic to check if text contains predominantly Hebrew, Arabic, or other RTL characters."""
    rtl_chars = re.compile(r'[֐-׿؀-ۿ]')
    if not text:
        return False
    # Check first 200 non-whitespace characters
    sample = ''.join(text.split())[:200]
    if not sample:
        return False
    rtl_count = len(rtl_chars.findall(sample))
    return (rtl_count / len(sample)) > 0.3

def add_styled_heading(doc, text, level, rtl=False):
    """Adds a heading with correct structure and alignment."""
    p = doc.add_heading(text, level=level)
    if rtl:
        set_rtl(p)
    return p

def add_styled_paragraph(doc, text, rtl=False):
    """Adds a standard paragraph with correct alignment."""
    p = doc.add_paragraph(text)
    if rtl:
        set_rtl(p)
    return p

def process_table(doc, table_html, rtl=False):
    """Recreates tables in DOCX preserving columns and rows."""
    rows_html = table_html.find_all('tr')
    if not rows_html:
        return

    # Determine table dimensions
    max_cols = 0
    for r_html in rows_html:
        cells = r_html.find_all(['td', 'th'])
        max_cols = max(max_cols, len(cells))

    if max_cols == 0:
        return

    # Create table in document
    docx_table = doc.add_table(rows=0, cols=max_cols)
    docx_table.style = 'Table Grid'

    for r_html in rows_html:
        row_cells_html = r_html.find_all(['td', 'th'])
        row_cells = docx_table.add_row().cells

        for i, cell_html in enumerate(row_cells_html):
            if i < len(row_cells):
                cell_text = cell_html.get_text(strip=True)
                row_cells[i].text = cell_text
                # Apply RTL context if necessary inside table cells
                if rtl or is_rtl_text(cell_text):
                    for p in row_cells[i].paragraphs:
                        set_rtl(p)

def process_image(doc, img_html, base_url, rtl=False):
    """Downloads and inserts an image inline, skipping potential company logos, social media, and UI assets."""
    src = img_html.get('src')
    if not src:
        return

    img_url = urljoin(base_url, src)

    # --- Identify and skip company logos and social media networks ---
    url_lower = img_url.lower()
    alt_text = (img_html.get('alt') or '').lower()
    class_names = " ".join(img_html.get('class') or []).lower()

    # Expanded lists of indicators commonly associated with logos, branding, header decoration, and social media
    logo_indicators = [
        'logo', 'brand', 'header-icon', 'header_icon', 'nav-logo', 'nav_logo', 'banner',
        'facebook', 'twitter', 'linkedin', 'instagram', 'youtube', 'pinterest', 'flickr',
        'social', 'share', 'icon', 'button', 'favicon', 'widget', 'nav-'
    ]

    is_logo = any(
        any(indicator in field for indicator in logo_indicators)
        for field in [url_lower, alt_text, class_names]
    )

    if is_logo:
        print(f"Skipping identified logo/social image (text match): {img_url}")
        return
    # ---------------------------------------------

    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(img_url, headers=headers, timeout=10)
        if response.status_code == 200:
            img_data = io.BytesIO(response.content)
            # Validate image with PIL
            with Image.open(img_data) as img:
                width, height = img.size

                # Size threshold: Skip tiny images, tracker pixels, and small social network icons (typically <= 40px)
                if width < 45 or height < 45:
                    print(f"Skipping UI icon/logo image (too small: {width}x{height}): {img_url}")
                    return

                # Skip extreme banner/divider proportions commonly used for branding lines/strips
                if (width / height > 8.0) or (height / width > 8.0):
                    print(f"Skipping UI icon/logo image (extreme aspect ratio): {img_url}")
                    return

                # Convert to RGB/RGBA if necessary to ensure compatibility with docx
                temp_img_io = io.BytesIO()
                img.save(temp_img_io, format=img.format if img.format else 'PNG')
                temp_img_io.seek(0)

                # Add image to doc
                p = doc.add_paragraph()
                if rtl:
                    p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
                else:
                    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

                run = p.add_run()
                # Constrain image width to fit page beautifully
                run.add_picture(temp_img_io, width=Inches(5.5))
    except Exception as e:
        print(f"Skipping image {img_url}: {e}")

def download_url_to_docx(url, output_dir):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    print(f"Fetching URL: {url}...")
    response = requests.get(url, headers=headers, timeout=15)
    if response.status_code != 200:
        raise Exception(f"Failed to load page. Status code: {response.status_code}")

    soup = BeautifulSoup(response.content, 'lxml')

    # Initialize Word Document
    doc = Document()

    # Try to extract the page title for the document metadata/title
    title_text = soup.title.string.strip() if soup.title else "Downloaded Document"
    clean_title = re.sub(r'[^\ -\u007E]+', '', title_text).strip()[:100]
    clean_title = re.sub(r'[\/:*?"<>|\n\r]', '', clean_title)

    # Add Source URL at the very top of the document
    p_url = doc.add_paragraph()
    p_url.add_run("Source URL: ").bold = True
    p_url.add_run(url)
    doc.add_paragraph("-" * 80)

    # Detect dominant text direction from body
    body_text = soup.body.get_text() if soup.body else ""
    is_rtl = is_rtl_text(body_text)
    if is_rtl:
        print("Detected Right-to-Left (RTL) writing direction.")

    # Process elements in order of appearance
    # Target semantic body content to avoid header/footer clutter where possible
    content_container = soup.find('main') or soup.find('article') or soup.body

    if not content_container:
        content_container = soup

    # We walk through children of the main content container
    for element in content_container.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'table', 'img']):
        # Skip nested duplicate parsing (e.g. standard images inside paragraphs can be handled inline or individually)
        if element.find_parent('table') and element.name != 'table':
            continue

        if element.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']:
            level = int(element.name[1])
            text = element.get_text(strip=True)
            if text:
                add_styled_heading(doc, text, level=min(level, 9), rtl=is_rtl)

        elif element.name == 'p':
            text = element.get_text(strip=True)
            if text:
                add_styled_paragraph(doc, text, rtl=is_rtl)

        elif element.name == 'table':
            process_table(doc, element, rtl=is_rtl)
            doc.add_paragraph()  # Space after table

        elif element.name == 'img':
            process_image(doc, element, url, rtl=is_rtl)

    # Save the file
    filename = f"{clean_title if clean_title else 'downloaded_page'}.docx"
    save_path = os.path.join(output_dir, filename)
    doc.save(save_path)
    print(f"Document successfully saved to: {save_path}")
    return save_path

### 3. Interactive Download Form
Input your target URL below and click **Run Cell** to start the download.

In [6]:
#@title Enter Webpage URL to Download
Target_URL = "https://www.fhwa.dot.gov/publications/research/infrastructure/pavements/ltpp/13092/001.cfm#toc384818105" #@param {type:"string"}

if Target_URL:
    try:
        saved_file = download_url_to_docx(Target_URL, output_folder)
    except Exception as e:
        print(f"An error occurred during processing: {e}")
else:
    print("Please provide a valid URL.")

Fetching URL: https://www.fhwa.dot.gov/publications/research/infrastructure/pavements/ltpp/13092/001.cfm#toc384818105...
Skipping UI icon/logo image (too small: 1x1): https://www.fhwa.dot.gov/images/1x1.gif
Skipping UI icon/logo image (too small: 274x40): https://www.fhwa.dot.gov/images/fhwa2/bg/imgdotfhwasm.png
Skipping identified logo/social image (text match): https://www.fhwa.dot.gov/images/fhwa2/icon/facebook_med.png
Skipping identified logo/social image (text match): https://www.fhwa.dot.gov/images/fhwa2/icon/youtube_med.png
Skipping identified logo/social image (text match): https://www.fhwa.dot.gov/images/fhwa2/icon/twitter_med.png
Skipping identified logo/social image (text match): https://www.fhwa.dot.gov/images/fhwa2/icon/flickr_med.png
Skipping identified logo/social image (text match): https://www.fhwa.dot.gov/images/fhwa2/icon/linkedin_med.png
Skipping identified logo/social image (text match): https://www.fhwa.dot.gov/images/fhwa2/icon/instagram_med.png
Document successf